In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [3]:
# datasets and dataloaders
from torch.utils.data import DataLoader
import  torchvision.transforms as  transforms

# image -> scale(0,1)->normalize(-1,+1)
transform=transforms.Compose([
    transforms.ToTensor(),# images into pytorch tensors and automatically scale
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# Training set
trainset= CIFAR10(root="./data", train=True,download=True,transform=transform)
testset= CIFAR10(root='./data',train=False,download=True,transform=transform)

100%|██████████| 170M/170M [31:01<00:00, 91.6kB/s]


In [4]:
# Convert it into dataloaders
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)


In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # First Convolution Layer
            # Input: 3 x 32 x 32
            # Output: 32 x 32 x 32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Output: 32 x 16 x 16

            # Second Convolution Layer
            # Output: 64 x 16 x 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Output: 64 x 8 x 8

            # Third Convolution Layer
            # Output: 128 x 8 x 8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Output: 128 x 4 x 4
        )

        self.fc_layers = nn.Sequential(
            # 128 * 4 * 4 = 2048
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),

            # Change 10 according to your number of classes
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)

        # Flatten
        x = x.view(x.size(0), -1)

        x = self.fc_layers(x)

        return x

In [6]:
model=CNN()

In [7]:
criterian= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters())

In [10]:
# Training the cnn model
epochs=10

for epoch in range(epochs):
  epoch_train_loss=0.0

  for images,labels in trainloader:
    # Forward propagation
    optimizer.zero_grad()
    output=model.forward(images)
    loss=criterian(output,labels) # loss fnx

    #computer gradients
    loss.backward()
    optimizer.step()

    epoch_train_loss+=loss.item()
  print(f"Epoch ->  {epoch+1}/{epochs} & lossValue -> {epoch_train_loss/len(trainloader)}")


Epoch ->  1/10 & lossValue -> 0.9059860083986732
Epoch ->  2/10 & lossValue -> 0.7337765209281536
Epoch ->  3/10 & lossValue -> 0.6138023694839014
Epoch ->  4/10 & lossValue -> 0.5161388731178116
Epoch ->  5/10 & lossValue -> 0.42079078122173125
Epoch ->  6/10 & lossValue -> 0.3424864224803722
Epoch ->  7/10 & lossValue -> 0.26559237714694894
Epoch ->  8/10 & lossValue -> 0.21285362630758597
Epoch ->  9/10 & lossValue -> 0.162941835511028
Epoch ->  10/10 & lossValue -> 0.14478179558044504


In [13]:
# Evaluation
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        # Forward pass
        outputs = model(images)

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        # Count correct predictions
        correct_labels += (predicted == labels).sum().item()

        # Count total labels
        total_labels += labels.size(0)

# Calculate accuracy
accuracy = (correct_labels / total_labels) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 75.54%
